In [293]:
import numpy as np
import pandas as pd
from astropy.io import fits

from astropy.time import Time, TimeDelta
import astropy.units as u

import random

In [294]:
df = pd.read_csv("/global/cfs/cdirs/desi/spectro/redux/daily/exposures-daily.csv") 

# these are the programs you can use to avoid manually typing the list all the time
program_b = ['bright', 'bright1b'] 
program_d = ['dark', 'dark1b'] 

"""
For extracting data. You can either extract data from a range of nights or a single night 
    (_range = True will give back data over a range ; _range = False will give data from the nights you provide
All data must be before January 1, 2026 (> 20260101) 
Honestly, you probably will only need a single date for timestamps, but in case you need a range or something, you have 
options!

Note: 
the input for the parameter "date" is in YYYYMMDD (ex. 20260821)
"""
def data_extraction(_date, _program=None, _range=False):
    mask = (df['SURVEY'] == 'main') & (~df['PROGRAM'].isin(['other', 'backup'])) & (df['EFFTIME_SPEC'] > 0)
    if (_range == True): 
        if type(_date) is list:
            mask &= (df['NIGHT'].between(_date[0], _date[1])) 
            # Returns data from the range of dates given 
        elif type(_date) is int:
            mask &= (df['NIGHT'].between(20260101, _date))
            # Returns data from the range from Jan 1, 2026 to the date given
    elif (_range == False): 
        if type(_date) is list:
            mask &= (df['NIGHT'].isin(_date))
            # Returns data from the each night given 
        elif type(_date) is int: 
             mask &= (df['NIGHT'] == _date)
            # Returns data from night given 
    _data = df[mask]
    # Masking the data for a single type of program: bright vs. dark
    if _program == None: data = _data
    else: data = _data[_data['PROGRAM'].isin(list(_program))]
    _tile, _exptime = data['TILEID'].unique(), (data.groupby('TILEID')['EXPTIME'].sum()).values 
        # Exposure times will be added to compensate multiple tiles being observed across different times 

    return _tile, _exptime, data # Returns the tile IDs, exposure time per tile for each night given, and all the data masked

In [295]:
"""
Exposure Ranges:
    BRIGHT program: [10, 85]
    DARK programs: [10, 78]

P.S.: Use one day since most days have a couple of times (some have multiple timestamps as well). Also, some days might not 
have a single bright/dark tile at all
NOTE (mostly reworded from Dylan): 
    1. The timestamp is when the tile STARTS
    2. You will notice that the timestamp given will not match the date that you input in. This is because the 'NIGHT' column 
    is to the current time at the TELESCOPE in mountain time (~ 7 hours behind UTC) and is set to the day the observing night 
    starts (i.e. NIGHT = 20260409 is the night of 4/9/2026 to morning of 4/10/2026). And it seems that a majority of the 
    timestamps are given on the morning of the next day 

"""
def tiles_list(date, pc_range, program, _range=False):
    t, expt, data = data_extraction(date, program, _range)
    print("GOOD BRIGHT TILES\n----") if program == program_b else print("GOOD DARK TILES\n----")
    if len(t)==0:
        print("No tiles were available for the nights given")
    else:
        print(f'Number of tiles available: {len(t)}\nTILE ID     TIMESTAMPS')
        tile_timestamps = {}
        # This is just to ensure the exposure ranges are up to date with the csv file since it updates frequently and the 
        # ranges change
        def range_calc(pc_range, program=None):
            t, expt, data = data_extraction(df['NIGHT'].max(), program, _range=True)
            return np.percentile(expt, pc_range)
        low, high = range_calc(pc_range, program)
        good_tiles, bad_tiles = t[(expt >= low) & (expt <= high)], t[(expt < low) | (expt > high)]
        
        def conversion(mjd):
            return Time(mjd, format = 'mjd', scale = 'utc').isot
        for tile in list(good_tiles):
            mjd_times = data.loc[(df['TILEID'] == tile), 'MJD']
            times_list = []
            for mjd in mjd_times: times_list.append(conversion(mjd))
            tile_timestamps[tile] = times_list
        if tile_timestamps == {}: 
            print('No tiles were within exposure range') 
        else: 
            for tiles, timestamps in tile_timestamps.items(): print(f'{tiles:<10} {timestamps}')
    print('\n\n')

In [296]:
date = 20261112 #example date
tiles_list(date, [10, 85], program_b)
tiles_list(date, [10, 78], program_d)

GOOD BRIGHT TILES
----
Number of tiles available: 3
TILE ID     TIMESTAMPS
120469     ['2026-11-13T01:35:01.008']
120469     ['2026-11-13T01:53:26.977']
120469     ['2026-11-13T08:56:44.686', '2026-11-13T09:13:23.987']



GOOD DARK TILES
----
Number of tiles available: 8
TILE ID     TIMESTAMPS
100902     ['2026-11-13T09:22:37.333', '2026-11-13T09:41:55.588']
100902     ['2026-11-13T09:59:52.863', '2026-11-13T10:28:34.942']





In [308]:
random_date = (df.loc[(df['NIGHT'] > 20260101, 'NIGHT')].sample()).item()
"""
if you just want random dates, you can use this random date generator above
"""
print(f'========\nStarting Day of Observation: {random_date}\n========')
tiles_list(random_date, [10, 85], program_b)
tiles_list(random_date, [10, 78], program_d)

Starting Day of Observation: 20260424
GOOD BRIGHT TILES
----
Number of tiles available: 5
TILE ID     TIMESTAMPS
121154     ['2026-04-25T03:14:47.490']
121154     ['2026-04-25T03:31:04.547']
121154     ['2026-04-25T09:18:11.238']
121154     ['2026-04-25T11:40:10.931']



GOOD DARK TILES
----
Number of tiles available: 4
TILE ID     TIMESTAMPS
100806     ['2026-04-25T09:41:48.174', '2026-04-25T10:00:58.078']
100806     ['2026-04-25T10:20:31.918', '2026-04-25T10:42:28.936']
100806     ['2026-04-25T10:55:29.009']
100806     ['2026-04-25T11:17:48.060']



